In [ ]:
#
# ⚡ UNIVERSAL FIRST CELL - Run this FIRST in every notebook!
# Compatible with: Google Colab, GitHub Codespaces, Local
#

import os
import sys

# Detect environment
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")

print(f"🚀 Environment: {'Colab' if IS_COLAB else 'Codespaces' if IS_CODESPACES else 'Local'}")

# Clone repo in Colab if needed
if IS_COLAB and not os.path.exists("computer-data-analysis-report"):
    print("Cloning repository...")
    !git clone --depth 1 https://github.com/Aidas-dev/computer-data-analysis-report.git

# Change to repo directory
repo_dir = "computer-data-analysis-report"
if os.path.exists(repo_dir):
    %cd {repo_dir}
    !git pull -q

# Install uv (faster than pip)
!pip install uv -q

# Install dependencies with uv
!uv pip install --system -r requirements.txt -q

# Setup DVC in Colab
if IS_COLAB:
    from google.colab import userdata
    try:
        access_key = userdata.get("OCI_ACCESS_KEY")
        secret_key = userdata.get("OCI_SECRET_KEY")
        !dvc remote modify --local oracle_remote access_key_id {access_key}
        !dvc remote modify --local oracle_remote secret_access_key {secret_key}
        !dvc pull -q
    except:
        pass

print("✅ Environment ready!")


# Machine Learning Pipeline: AI Data Center Feasibility

This notebook provides a template for training machine learning models (Scikit-Learn and XGBoost) to classify the success probability of AI data center projects.

**🔥 Colab GPU Setup:**
To take full advantage of XGBoost's speed, turn on the free Google Colab GPU:
1. Go to the top menu: **Runtime > Change runtime type**
2. Under Hardware accelerator, select **T4 GPU** and click Save.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

print(f"XGBoost version: {xgb.__version__}")

### 1. Load Data
*(For this example, we generate a synthetic dataset mimicking the parameters outlined in our `docs/research_foundation.md`)*

In [ ]:
# Generate synthetic project data
np.random.seed(42)
n_samples = 5000

data = pd.DataFrame({
    'project_scale_mw': np.random.uniform(10, 500, n_samples),  # Megawatt scale
    'firm_market_cap_bln': np.random.uniform(5, 3000, n_samples), # Firm financial strength
    'interest_rate_pct': np.random.uniform(2.0, 7.5, n_samples),  # Economic climate
    'grid_queue_delay_months': np.random.exponential(12, n_samples), # Infrastructure hurdle
})

# Create target variable: 1 = Success (Built), 0 = Failed/Withdrawn
# Logic: Larger firms, lower interest rates, and shorter grid delays increase success odds.
success_prob = (
    (data['firm_market_cap_bln'] / 3000) * 0.4 +
    ((7.5 - data['interest_rate_pct']) / 5.5) * 0.3 +
    (np.exp(-data['grid_queue_delay_months'] / 24)) * 0.3
)

data['success'] = (success_prob + np.random.normal(0, 0.1, n_samples) > 0.5).astype(int)

print("Synthetic Data Center Projects Generated:")
display(data.head())
print("\nClass Distribution:\n", data['success'].value_counts(normalize=True))

### 2. Preprocessing & Train-Test Split

In [ ]:
X = data.drop('success', axis=1)
y = data['success']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

### 3. Baseline Model: Scikit-Learn Random Forest
A standard CPU-bound ensemble tree model.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
print("=== Random Forest Results ===")
print(f"Accuracy: {accuracy_score(y_test, rf_preds):.4f}")
print(classification_report(y_test, rf_preds))

### 4. Advanced Model: GPU-Accelerated XGBoost
XGBoost is the industry standard for tabular data. By setting `device="cuda"` (if a GPU is available) or `tree_method="hist"`, it trains orders of magnitude faster on massive datasets.

In [ ]:
# Determine if a GPU is available in the Colab session
# If no GPU, it gracefully falls back to highly-optimized CPU training (hist)
try:
    # Test for CUDA availability
    import subprocess
    subprocess.check_output('nvidia-smi')
    device = "cuda"
    print("✅ GPU detected! Training on CUDA.")
except Exception:
    device = "cpu"
    print("⚠️ No GPU detected. Falling back to CPU (hist). Turn on T4 GPU in Colab Runtime settings for speed!")

# Initialize XGBoost Classifier (>= 2.0.0 uses 'device' parameter)
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    device=device,
    random_state=42
)

# Train Model
xgb_model.fit(X_train, y_train)

# Evaluate
xgb_preds = xgb_model.predict(X_test)
print("\n=== XGBoost Results ===")
print(f"Accuracy: {accuracy_score(y_test, xgb_preds):.4f}")
print(classification_report(y_test, xgb_preds))

### 5. Feature Importance
Understanding what parameters actually influenced the model's predictions.

In [ ]:
# Extract feature importances from the XGBoost model
importances = pd.Series(
    xgb_model.feature_importances_, 
    index=X.columns
).sort_values(ascending=False)

print("Feature Importances (Impact on Project Success):")
display(importances.to_frame(name="Importance Score"))